# Pump It Up: Data Mining the Water Table

Team members: _add names here_

Predict whether each Tanzanian waterpoint is functional, needs repair, or is non-functional. Explain who could use these predictions and their practical limits. This notebook carries over the source notebook’s exploratory plots, feature engineering, eight model experiments, holdout evaluation, inspection-policy scenario, and submission generation.

## 1. Setup

Install `requirements.txt` first. The default quick run uses 9,000 development rows and three folds. For final results, set `QUICK_RUN=False` and `RUN_FINAL_EVALUATION=True`, restart the kernel, and run all cells. The full grid search can take considerable time.

In [ ]:
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import xgboost

from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GridSearchCV
)

from sklearn.metrics import (
    accuracy_score, f1_score, recall_score, make_scorer,
    classification_report, confusion_matrix
)

from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier, BaggingClassifier
)

from xgboost import XGBClassifier

SEED = 42

# False uses all 47,520 development rows.
# True uses 9,000 development rows for a quicker trial.
QUICK_RUN = True
N_SPLITS = 3 if QUICK_RUN else 5

# Choose the selection metric before comparing models.
PRIMARY_METRIC = "accuracy"

# Keep final evaluation disabled while experimenting.
# When finished choosing the model, enable this and rerun the notebook.
RUN_FINAL_EVALUATION = False

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data" / "raw").is_dir():
    REPO_ROOT = REPO_ROOT.parent
DATA_DIR = REPO_ROOT / "data" / "raw"
if not DATA_DIR.is_dir():
    raise FileNotFoundError("Run from the repository root or notebooks/.")
OUTPUT_DIR = REPO_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

CLASS_NAMES = [
    "functional",
    "functional needs repair",
    "non functional"
]

# Numeric labels are required by XGBoost.
# All models will use the same encoding.
label_to_number = {name: number for number, name in enumerate(CLASS_NAMES)}
number_to_label = {number: name for name, number in label_to_number.items()}

sns.set_theme(style="whitegrid")
sklearn.set_config(working_memory=128)

print("scikit-learn:", sklearn.__version__)
print("XGBoost:", xgboost.__version__)
print("Quick run:", QUICK_RUN)
print("CV folds:", N_SPLITS)


## 2. Load and align the data

Join labels by `id`. Keep the unlabeled competition test set out of validation.

In [ ]:
# Merge using ID so every pump receives the correct label.

train_values = pd.read_csv(
    DATA_DIR / "train_values.csv"
)
train_labels = pd.read_csv(
    DATA_DIR / "train_labels.csv"
)
test_values = pd.read_csv(
    DATA_DIR / "test_values.csv"
)
submission_format = pd.read_csv(DATA_DIR / "submission_format.csv")

# Check the IDs before joining or making predictions.
for frame in [train_values, train_labels, test_values, submission_format]:
    assert frame["id"].is_unique

assert set(train_values["id"]) == set(train_labels["id"])
assert set(test_values["id"]) == set(submission_format["id"])
assert set(train_values["id"]).isdisjoint(test_values["id"])
assert set(train_values.columns) == set(test_values.columns)

df = train_values.merge(
    train_labels, on="id", how="left", validate="one_to_one"
)

assert df["status_group"].notna().all()
assert set(df["status_group"]) == set(CLASS_NAMES)

# Remove the target and identifier from the model inputs.
X = df.drop(columns=["id", "status_group"])

# Encoding changes the label representation, not the classification task.
# 0 = functional, 1 = needs repair, 2 = non functional.
y = df["status_group"].map(label_to_number).astype(int)

X_test = test_values.drop(columns="id")

print("Training inputs:", X.shape)
print("Training target:", y.shape)
print("Competition test inputs:", X_test.shape)

display(df.head())


## 3. Reserve a holdout set

Make feature and model decisions with the development set. Evaluate on the holdout once at the end.

In [ ]:
# Stratification preserves class proportions in both sets.
# Use development data for all feature and model decisions.

X_dev, X_holdout, y_dev, y_holdout = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=SEED
)

# Keep readable class names in the exploration dataframe.
eda = X_dev.copy()
eda["status_group"] = y_dev.map(number_to_label)

print("Development rows:", len(X_dev))
print("Holdout rows:", len(X_holdout))

class_summary = eda["status_group"].value_counts().to_frame("count")
class_summary["percent"] = 100 * class_summary["count"] / len(eda)

display(class_summary)


## 4. Explore the development data

Add interpretations of class balance, missingness, category patterns, and geographic patterns alongside the plots.

In [ ]:
# Imbalance motivates stratified CV and reporting macro-F1.
# Missingness helps us decide which imputation steps are needed.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(
    data=eda,
    y="status_group",
    order=CLASS_NAMES,
    color="steelblue",
    ax=axes[0]
)
axes[0].set(title="Development class counts", xlabel="Pumps", ylabel="")

missing = X_dev.isna().mean().mul(100).sort_values(ascending=False)
missing = missing[missing > 0]

sns.barplot(
    x=missing.values,
    y=missing.index,
    color="coral",
    ax=axes[1]
)
axes[1].set(title="Missing values", xlabel="Percent missing", ylabel="")

plt.tight_layout()
plt.show()

# Inspect cardinality before deciding which categorical fields to retain.
display(pd.DataFrame({
    "unique_values": X_dev.nunique(),
    "missing_percent": X_dev.isna().mean() * 100
}))


In [ ]:
# Row percentages make differently sized categories easier to compare.
# Always check category counts before interpreting extreme percentages.

for feature in ["quantity", "waterpoint_type"]:
    proportions = pd.crosstab(
        eda[feature],
        eda["status_group"],
        normalize="index"
    ).reindex(columns=CLASS_NAMES, fill_value=0)

    plt.figure(figsize=(10, 4))

    sns.heatmap(
        proportions,
        annot=True,
        fmt=".1%",
        cmap="Blues",
        vmin=0,
        vmax=1
    )

    plt.title(f"Pump status within each {feature}")
    plt.xlabel("Pump status")
    plt.tight_layout()
    plt.show()

    display(eda[feature].value_counts().to_frame("pumps"))


## 5. Prepare features

Row-wise transformations apply to every split. Imputation and encoding are fitted inside each cross-validation fold.

In [ ]:
# These rules use individual rows, not statistics learned from other rows.
# Imputation and encoding will be learned inside each CV fold.

DROP_COLUMNS = [
    "recorded_by",        # Constant in these data.
    "wpt_name",          # Many nearly unique names.
    "subvillage",        # High cardinality; excluded in the initial model.
    "scheme_name",       # High cardinality and substantial missingness.
    "ward",              # Can be added back in a later CV experiment.
    "quantity_group",    # Duplicate of quantity.
    "extraction_type_group",
    "extraction_type_class",
    "management_group",
    "payment_type",
    "quality_group",
    "source_type",
    "source_class",
    "waterpoint_type_group"
]

# Verify the two exact redundancy assumptions using development data.
assert X_dev["recorded_by"].nunique(dropna=False) == 1
assert X_dev["quantity"].equals(X_dev["quantity_group"])

def engineer_features(frame):
    data = frame.copy()

    # Year 0 is invalid; construction after observation is inconsistent.
    recorded = pd.to_datetime(data["date_recorded"], errors="coerce")
    year = pd.to_numeric(data["construction_year"], errors="coerce")
    year = year.mask((year <= 0) | (year > recorded.dt.year))

    data["construction_year"] = year
    data["pump_age"] = recorded.dt.year - year
    data["record_year"] = recorded.dt.year

    # Cyclic month features place December near January.
    month = recorded.dt.month
    data["month_sin"] = np.sin(2 * np.pi * (month - 1) / 12)
    data["month_cos"] = np.cos(2 * np.pi * (month - 1) / 12)

    # Flag implausible or missing coordinate pairs.
    bad_coordinates = (
        data["longitude"].eq(0)
        | np.isclose(data["latitude"], 0, atol=1e-6)
        | data[["longitude", "latitude"]].isna().any(axis=1)
    )

    data["coordinates_missing"] = bad_coordinates.astype(int)
    data.loc[bad_coordinates, ["longitude", "latitude"]] = np.nan

    # Preserve ambiguous zeros and give models explicit zero indicators.
    for column in ["population", "gps_height", "amount_tsh"]:
        data[column + "_is_zero"] = data[column].eq(0).astype(int)

    # Compress skewed nonnegative measurements for KNN and logistic regression.
    for column in ["population", "amount_tsh"]:
        data[column + "_log1p"] = np.log1p(data[column].clip(lower=0))

    # Administrative codes are category labels, not numeric distances.
    data["district_id"] = (
        data["region_code"].astype(str)
        + "_"
        + data["district_code"].astype(str)
    )

    data = data.drop(
        columns=DROP_COLUMNS + [
            "date_recorded", "region_code", "district_code",
            "population", "amount_tsh"
        ]
    )

    # Standardize text while preserving genuinely missing values.
    categorical = data.select_dtypes(
        include=["object", "string", "bool"]
    ).columns

    for column in categorical:
        data[column] = data[column].map(
            lambda value: str(value).strip().lower()
            if pd.notna(value) else np.nan
        )
        data[column] = data[column].replace("", np.nan)

    return data

X_dev_clean = engineer_features(X_dev)
X_holdout_clean = engineer_features(X_holdout)
X_test_clean = engineer_features(X_test)

display(X_dev_clean.head())
print("Prepared development shape:", X_dev_clean.shape)


In [ ]:
# The age plot excludes missing ages automatically.
# Spatial patterns also warn that random CV may overestimate
# performance when deploying in entirely new regions.

plot_data = X_dev_clean.copy()
plot_data["status_group"] = y_dev.map(number_to_label)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.boxplot(
    data=plot_data,
    x="pump_age",
    y="status_group",
    order=CLASS_NAMES,
    showfliers=False,
    color="lightblue",
    ax=axes[0]
)
axes[0].set(title="Age at recording", ylabel="")

sample = plot_data.sample(
    n=min(5000, len(plot_data)),
    random_state=SEED
)

sns.scatterplot(
    data=sample,
    x="longitude",
    y="latitude",
    hue="status_group",
    hue_order=CLASS_NAMES,
    alpha=0.4,
    s=12,
    ax=axes[1]
)
axes[1].set_title("Location and pump status")

plt.tight_layout()
plt.show()


## 6. Compare models with cross-validation

All eight experiments use the same stratified folds. Accuracy selects a model; macro-F1 and repair-class recall give more detail. Explain the trade-offs and results.

In [ ]:
# All learned preprocessing is fitted inside the training fold.
# Numeric scaling is used only for KNN and multinomial logistic regression.

numeric_columns = X_dev_clean.select_dtypes(include="number").columns.tolist()
categorical_columns = X_dev_clean.columns.difference(numeric_columns).tolist()

def make_pipeline(model, scale_numeric=False):
    numeric_steps = [
        ("impute", SimpleImputer(strategy="median", add_indicator=True))
    ]

    if scale_numeric:
        numeric_steps.append(("scale", StandardScaler()))

    categorical_steps = Pipeline([
        ("impute", SimpleImputer(
            strategy="constant",
            fill_value="__missing__"
        )),
        ("encode", OneHotEncoder(
            handle_unknown="infrequent_if_exist",
            min_frequency=20,
            max_categories=50,
            sparse_output=True
        ))
    ])

    # Keep the original category limits for the initial expanded comparison.
    # Later, test min_frequency=10 and max_categories=None as a separate
    # experiment using the same rows and folds.

    preprocess = ColumnTransformer([
        ("numeric", Pipeline(numeric_steps), numeric_columns),
        ("categorical", categorical_steps, categorical_columns)
    ], sparse_threshold=1.0)

    return Pipeline([
        ("preprocess", preprocess),
        ("model", model)
    ])


In [ ]:
# Every model receives the same rows and folds for a fair comparison.
# Repair recall measures how many actual repair cases are detected.

if QUICK_RUN:
    X_fit, _, y_fit, _ = train_test_split(
        X_dev_clean,
        y_dev,
        train_size=9000,
        stratify=y_dev,
        random_state=SEED
    )
else:
    X_fit = X_dev_clean
    y_fit = y_dev

cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

folds = list(cv.split(X_fit, y_fit))

scoring = {
    "accuracy": "accuracy",
    "macro_f1": "f1_macro",
    "repair_recall": make_scorer(
        recall_score,
        labels=[1],        # Class 1 means functional needs repair.
        average="macro",
        zero_division=0
    )
}

assert PRIMARY_METRIC in scoring

print("Rows used by every model:", len(X_fit))
print("CV folds:", len(folds))
print("Selection metric:", PRIMARY_METRIC)


In [ ]:
# Bagging trains decision trees on bootstrap samples.
# Random forest also randomizes feature selection at individual splits.
# XGBoost builds boosted trees for the three-class task.

experiments = {
    "Majority baseline": (
        make_pipeline(DummyClassifier(strategy="most_frequent")),
        {}
    ),

    "KNN": (
        make_pipeline(
            KNeighborsClassifier(algorithm="brute", n_jobs=2),
            scale_numeric=True
        ),
        {
            "model__n_neighbors": [11, 31],
            "model__weights": ["uniform", "distance"]
        }
    ),

    "Multinomial logistic regression": (
        make_pipeline(
            # With three classes, lbfgs optimizes multinomial loss.
            # Do not add the deprecated multi_class argument.
            LogisticRegression(
                solver="lbfgs",
                max_iter=3000
            ),
            scale_numeric=True
        ),
        {
            "model__C": [0.1, 1.0],
            "model__class_weight": [None, "balanced"]
        }
    ),

    "Decision tree": (
        make_pipeline(
            DecisionTreeClassifier(random_state=SEED)
        ),
        {
            "model__max_depth": [10, 20, None],
            "model__min_samples_leaf": [2, 10]
        }
    ),

    "Random forest": (
        make_pipeline(
            RandomForestClassifier(
                n_estimators=150,
                random_state=SEED,
                n_jobs=2
            )
        ),
        {
            "model__max_features": ["sqrt", 0.5],
            "model__min_samples_leaf": [1, 5]
        }
    ),

    "Extra Trees": (
        make_pipeline(
            ExtraTreesClassifier(
                n_estimators=150,
                random_state=SEED,
                n_jobs=2
            )
        ),
        {
            "model__max_features": ["sqrt", 0.5],
            "model__min_samples_leaf": [1, 5]
        }
    ),

    "Bagging trees": (
        make_pipeline(
            BaggingClassifier(
                estimator=DecisionTreeClassifier(random_state=SEED),
                n_estimators=100,
                bootstrap=True,
                random_state=SEED,
                n_jobs=2
            )
        ),
        {
            "model__max_samples": [0.7, 1.0],
            "model__estimator__min_samples_leaf": [2, 5]
        }
    ),

    "XGBoost": (
        make_pipeline(
            XGBClassifier(
                objective="multi:softprob",
                num_class=3,
                eval_metric="mlogloss",
                tree_method="hist",
                n_estimators=250,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=SEED,
                n_jobs=2
            )
        ),
        {
            "model__max_depth": [4, 7],
            "model__learning_rate": [0.05, 0.1]
        }
    )
}

print("Models:", list(experiments))


In [ ]:
# Searches run one at a time to control memory use.
# Training scores are omitted because they add computation and are
# particularly misleading for distance-weighted KNN.
# If logistic regression raises a convergence warning, increase max_iter.

searches = {}
rows = []

for name, (pipeline, parameters) in experiments.items():
    print(f"\nStarting: {name}", flush=True)
    start = perf_counter()

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=parameters,
        scoring=scoring,
        refit=PRIMARY_METRIC,
        cv=folds,
        n_jobs=1,
        return_train_score=False,
        error_score="raise",
        verbose=1
    )

    search.fit(X_fit, y_fit)
    searches[name] = search

    # Extract all metrics for the configuration selected by the primary metric.
    best = search.best_index_
    scores = search.cv_results_

    row = {
        "model": name,
        "search_seconds": perf_counter() - start,
        "best_parameters": search.best_params_
    }

    for metric in scoring:
        row["cv_" + metric] = scores["mean_test_" + metric][best]
        row["sd_" + metric] = scores["std_test_" + metric][best]

    rows.append(row)

    # Save every configuration, not only the winning one.
    filename = name.lower().replace(" ", "_") + "_search.csv"
    pd.DataFrame(scores).to_csv(OUTPUT_DIR / filename, index=False)

    print("Best parameters:", search.best_params_, flush=True)

# Select by the declared primary metric, with a secondary tie-breaker.
primary_column = "cv_" + PRIMARY_METRIC
secondary_column = (
    "cv_macro_f1" if PRIMARY_METRIC != "macro_f1" else "cv_accuracy"
)

results = pd.DataFrame(rows).sort_values(
    [primary_column, secondary_column],
    ascending=False
).reset_index(drop=True)

display(results)
results.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)

# CV selection scores may be optimistic after searching many configurations.
# Final evaluation is kept separate.
winner_name = results.loc[0, "model"]
winner = clone(searches[winner_name].best_estimator_)

print("Selected model:", winner_name)


In [ ]:
# Accuracy, macro-F1, and repair recall answer different questions.
# Fold standard deviations are variability measures, not confidence intervals.

fig, axes = plt.subplots(1, 3, figsize=(19, 6))

metrics = [
    ("cv_accuracy", "CV accuracy"),
    ("cv_macro_f1", "CV macro-F1"),
    ("cv_repair_recall", "CV repair-class recall")
]

for ax, (metric, title) in zip(axes, metrics):
    sns.barplot(
        data=results,
        x=metric,
        y="model",
        order=results["model"].tolist(),
        color="steelblue",
        ax=ax
    )

    ax.set(title=title, xlabel="Score", ylabel="", xlim=(0, 1))

plt.tight_layout()
plt.show()

# Stop here while changing features or tuning settings.
# For final results, set QUICK_RUN=False and RUN_FINAL_EVALUATION=True
# in the settings cell, restart the kernel, and run all cells.


## 7. Final holdout evaluation

Run after model selection. Report class-specific errors and avoid retuning against this holdout.

In [ ]:
# Run this only after completing model selection on the full development set.
# Do not use the holdout results to repeatedly retune the model.

def predict_in_batches(model, frame, batch_size=500):
    predictions = []

    for start in range(0, len(frame), batch_size):
        batch = frame.iloc[start:start + batch_size]
        predictions.append(model.predict(batch))

    return np.concatenate(predictions).astype(int)

# This flag prevents later cells from using an old holdout prediction.
holdout_evaluated = False

if QUICK_RUN or not RUN_FINAL_EVALUATION:
    print("Final evaluation skipped. Finish model selection first.")
else:
    # Confirm the searches used the full development data.
    assert len(X_fit) == len(X_dev_clean)

    winner.fit(X_dev_clean, y_dev)
    holdout_pred = predict_in_batches(winner, X_holdout_clean)

    baseline = clone(searches["Majority baseline"].best_estimator_)
    baseline.fit(X_dev_clean, y_dev)
    baseline_pred = predict_in_batches(baseline, X_holdout_clean)

    final_rows = []

    for name, prediction in [
        (winner_name, holdout_pred),
        ("Majority baseline", baseline_pred)
    ]:
        final_rows.append({
            "model": name,
            "holdout_accuracy": accuracy_score(y_holdout, prediction),
            "holdout_macro_f1": f1_score(
                y_holdout, prediction, average="macro"
            )
        })

    final_scores = pd.DataFrame(final_rows)
    display(final_scores)
    final_scores.to_csv(OUTPUT_DIR / "holdout_scores.csv", index=False)

    print(classification_report(
        y_holdout,
        holdout_pred,
        labels=[0, 1, 2],
        target_names=CLASS_NAMES,
        zero_division=0
    ))

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    for ax, normalization, title, number_format in [
        (axes[0], None, "Holdout: counts", "d"),
        (axes[1], "true", "Holdout: proportions within actual class", ".1%")
    ]:
        matrix = confusion_matrix(
            y_holdout,
            holdout_pred,
            labels=[0, 1, 2],
            normalize=normalization
        )

        sns.heatmap(
            matrix,
            annot=True,
            fmt=number_format,
            cmap="Blues",
            xticklabels=CLASS_NAMES,
            yticklabels=CLASS_NAMES,
            ax=ax
        )

        ax.set(title=title, xlabel="Predicted", ylabel="Actual")
        ax.tick_params(axis="x", rotation=25)

    plt.tight_layout()
    plt.show()

    holdout_evaluated = True


## 8. Inspection-policy scenario

The costs below are hypothetical illustration values, not measured financial savings.

In [ ]:
# Flag pumps predicted as needing repair or being non-functional.
# This policy does not change the original three-class model.
# The cost values below are hypothetical, not measured savings.

if not holdout_evaluated:
    print("Complete the final holdout evaluation first.")
else:
    actual_attention = y_holdout.to_numpy() != 0
    flagged = holdout_pred != 0

    true_positive = np.sum(flagged & actual_attention)
    false_positive = np.sum(flagged & ~actual_attention)
    false_negative = np.sum(~flagged & actual_attention)

    inspection_yield = true_positive / max(flagged.sum(), 1)
    coverage = true_positive / max(actual_attention.sum(), 1)

    display(pd.Series({
        "pumps_flagged": int(flagged.sum()),
        "attention_cases_found": int(true_positive),
        "functional_pumps_flagged": int(false_positive),
        "attention_cases_missed": int(false_negative),
        "inspection_yield": inspection_yield,
        "coverage": coverage,
        "random_inspection_expected_yield": actual_attention.mean()
    }))

    # Arbitrary currency units; replace with justified assumptions if available.
    inspection_cost = 20
    missed_case_penalty = 100

    model_cost = (
        flagged.sum() * inspection_cost
        + false_negative * missed_case_penalty
    )
    inspect_all_cost = len(y_holdout) * inspection_cost

    print("Hypothetical model-policy cost:", model_cost)
    print("Hypothetical inspect-all cost:", inspect_all_cost)
    print(
        "Hypothetical difference:",
        inspect_all_cost - model_cost
    )

    # Positive difference favors the model under these assumptions.
    # This scenario excludes repair costs, travel, and actual field outcomes.


## 9. Refit and write a submission

After the holdout analysis, refit on all labeled rows. The output is saved to ignored `outputs/submission.csv` in the template ID order.

In [ ]:
# After recording holdout performance, use all 59,400 labeled pumps.
# Convert numeric predictions back to the exact required class names.

if not holdout_evaluated:
    print("Submission skipped until final evaluation is complete.")
else:
    final_model = clone(winner)
    final_model.fit(engineer_features(X), y)

    test_predictions = predict_in_batches(final_model, X_test_clean)

    predictions_by_id = pd.Series(
        test_predictions,
        index=test_values["id"]
    ).map(number_to_label)

    # Map by ID rather than assuming the template has the same row order.
    submission = submission_format[["id"]].copy()
    submission["status_group"] = submission["id"].map(predictions_by_id)

    assert submission.columns.tolist() == ["id", "status_group"]
    assert len(submission) == len(test_values)
    assert submission["id"].equals(submission_format["id"])
    assert submission["status_group"].notna().all()
    assert set(submission["status_group"]).issubset(CLASS_NAMES)

    submission_path = OUTPUT_DIR / "submission.csv"
    submission.to_csv(submission_path, index=False)

    display(submission.head())
    print("Saved:", submission_path)


## 10. Conclusions

Write the team’s findings here: best model, score variability, class-specific limitations, possible geographic shift, practical usefulness, and next steps.